In [1]:
import torch
from torch.utils.data import Dataset
import os

class BeatmapChunkDataset(Dataset):
    def __init__(self, input_folder):
        self.audio_folder = os.path.join(input_folder, "audio")
        
        df = pd.read_csv(os.path.join(input_folder, "chunked.csv"))
        self.groups = list(df.groupby(["id", "chunk_id"]))

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        (beatmap_id, chunk_id), group = self.groups[idx]

        features = torch.tensor(
            group[["type_circle", "type_slider", "type_spinner", 
                   "hit_start_rel", "hit_end_rel"]].values, dtype=torch.float32)
        beatmapset_id = beatmap_id.split("-")[0]
        chunk_audio_path = os.path.join(self.audio_folder, f"{beatmapset_id}_chunk{chunk_id}.pt")
        
        difficulty_rating = torch.tensor([group.iloc[0]["difficulty_rating"]], dtype=torch.float32)
        
        return {
            "beatmap_id": beatmap_id,
            "chunk_id": chunk_id,
            "features": features,
            "audio": torch.load(chunk_audio_path),
            "difficulty_rating": difficulty_rating 
        }


In [2]:
import torch
from torch.nn.utils.rnn import pad_sequence

def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
    """
    Causal mask for decoder self-attention (prevents attending to future positions).
    Shape: [sz, sz]
    """
    mask = torch.triu(torch.ones(sz, sz, dtype=torch.bool), diagonal=1)
    return mask  # True = masked

def collate_fn(batch):
    beatmap_ids = [item["beatmap_id"] for item in batch]
    chunk_ids = [item["chunk_id"] for item in batch]

    features_list = [item["features"] for item in batch]  # list of [seq_len_i, feature_dim]
    features_padded = pad_sequence(features_list, batch_first=True, padding_value=0.0)  # [B, max_seq_len_tgt, feat_dim]

    tgt_key_padding_mask = torch.zeros(features_padded.shape[:2], dtype=torch.bool)
    for i, feat in enumerate(features_list):
        tgt_key_padding_mask[i, :feat.shape[0]] = 1

    max_tgt_len = features_padded.size(1)
    tgt_causal_mask = generate_square_subsequent_mask(max_tgt_len)  # [max_tgt_len, max_tgt_len]

    audio_list = [item["audio"].squeeze(0) for item in batch]
    audio_padded = pad_sequence(audio_list, batch_first=True, padding_value=0.0)  # [B, max_seq_len_audio, 768]

    audio_mask = torch.zeros(audio_padded.shape[:2], dtype=torch.bool)
    for i, a in enumerate(audio_list):
        audio_mask[i, :a.shape[0]] = 1

    difficulty_ratings = torch.tensor([item["difficulty_rating"] for item in batch], dtype=torch.float).unsqueeze(1)

    return {
        "beatmap_ids": beatmap_ids,
        "chunk_ids": torch.tensor(chunk_ids, dtype=torch.long),
        "features": features_padded,
        "tgt_key_padding_mask": tgt_key_padding_mask,  # [B, max_tgt_len]
        "tgt_causal_mask": tgt_causal_mask,      # [max_tgt_len, max_tgt_len]
        "audio": audio_padded,                   # encoder input
        "audio_mask": audio_mask,                # [B, max_seq_len_audio]
        "difficulty_rating": difficulty_ratings
    }


In [28]:
import pandas as pd
from torch.utils.data import DataLoader

input_folder = "/home/saliherdemk/try_dataset/chunked"
dataset = BeatmapChunkDataset(input_folder)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn = collate_fn)

for batch in dataloader:
    # print(batch['audio'], batch["audio_mask"])
    print(batch["features"].shape, batch["tgt_key_padding_mask"].shape,  batch["tgt_causal_mask"])
    break


torch.Size([2, 40, 5]) torch.Size([2, 40]) tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])


# Model

In [9]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim=768, emb_dim=1023, max_seq_len=1000, nhead=8, num_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=256, kernel_size=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=emb_dim, kernel_size=1)

        d_model = emb_dim + 1

        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, d_model))  # [1, seq_len, d_model]
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x, diff_rating, audio_mask):
        # x: [batch, seq_len, d_model]
        x = x.permute(0, 2, 1)  # [batch, d_model, seq_len]
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x) # [batch, d_model, seq_len]
        x = x.permute(0, 2, 1) # [batch, seq_len, d_model]
    
        seq_len = x.size(1)
        diff_rating = diff_rating.unsqueeze(1).expand(-1, seq_len, -1)
        x = torch.cat([x, diff_rating], dim=-1)

        x = x + self.pos_embedding[:, :seq_len, :]
        out = self.encoder(x, src_key_padding_mask=audio_mask)
    
        return out



In [11]:
encoder = Encoder()
for batch in dataloader:
    audio_batch = batch["audio"]
    audio_mask = batch["audio_mask"]
    diff_batch = batch["difficulty_rating"]
    memory = encoder(audio_batch, diff_batch, ~audio_mask)
    
    tgt = batch["features"]

    tgt_casual_mask = batch["tgt_causal_mask"]
    tgt_key_padding_mask = batch["tgt_key_padding_mask"]
    print(tgt.shape)
    print(tgt_casual_mask.shape)
    print(tgt_key_padding_mask.shape)
    print(audio_mask.shape)
    break

torch.Size([2, 50, 5])
torch.Size([50, 50])
torch.Size([2, 50])
torch.Size([2, 749])


In [12]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(
        self,
        d_model=1024,
        num_types=3,
        max_seq_len=1000,
        nhead=8,
        num_layers=6,
        dim_feedforward=2048,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(5, d_model)  # 3 type + 2 timing

        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, d_model))

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.type_head = nn.Linear(d_model, num_types)
        self.cont_head = nn.Linear(d_model, 2)

    def forward(self, tgt, memory, tgt_causal_mask, tgt_key_padding_mask, memory_key_padding_mask):
        x = self.input_proj(tgt.float())

        seq_len = x.size(1)
        x = x + self.pos_embedding[:, :seq_len, :]

        out = self.decoder(
            tgt = x,
            memory = memory,
            tgt_mask = tgt_causal_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask)

        type_logits = self.type_head(out)
        cont_preds = self.cont_head(out)

        return type_logits, cont_preds


In [20]:
encoder = Encoder()
decoder = Decoder()
for batch in dataloader:
    audio_batch = batch["audio"]
    audio_mask = batch["audio_mask"]
    diff_batch = batch["difficulty_rating"]
    memory = encoder(audio_batch, diff_batch, ~audio_mask)
    
    tgt = batch["features"]

    tgt_casual_mask = batch["tgt_causal_mask"]
    tgt_key_padding_mask = batch["tgt_key_padding_mask"]
    print(tgt.shape)
    print(tgt_casual_mask.shape)
    print(tgt_key_padding_mask.shape)
    print(audio_mask.shape)
    
    
    o = decoder(tgt, memory, tgt_casual_mask, ~tgt_key_padding_mask, ~audio_mask)
    print(o[0].shape, o[1].shape)
    break

torch.Size([2, 53, 5])
torch.Size([53, 53])
torch.Size([2, 53])
torch.Size([2, 749])
torch.Size([2, 53, 3]) torch.Size([2, 53, 2])


In [27]:
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)

for batch in dataloader:
    audio = batch["audio"]
    audio_mask = batch["audio_mask"]
    diff = batch["difficulty_rating"]
    features = batch["features"]
    tgt_causal_mask = batch["tgt_causal_mask"]
    tgt_key_padding_mask = batch["tgt_key_padding_mask"]

    memory = encoder(audio, diff, ~audio_mask)
    type_logits, cont_preds = decoder(
        features, memory,
        tgt_causal_mask=tgt_causal_mask,
        tgt_key_padding_mask=~tgt_key_padding_mask,
        memory_key_padding_mask=~audio_mask
    )

    type_target = features[..., :3].argmax(dim=-1)
    cont_target = features[..., 3:]

    loss_type = nn.CrossEntropyLoss()(type_logits.view(-1, 3), type_target.view(-1))
    loss_cont = nn.MSELoss()(cont_preds, cont_target)
    loss = loss_type + loss_cont

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    break

# print(f"Epoch {epoch} loss: {loss.item():.4f}")
